In [2]:
#packages and modules

from pyspark.sql.session import SparkSession
import pyspark.sql.functions as f

import os

from dotenv import load_dotenv

env = load_dotenv('/mnt/d/linux/project/vars/.env')

In [3]:
#variables

pgconn = os.getenv('POSTGRE_CONN')
pgdriver = os.getenv('PG_DRIVER')
pguser = os.getenv('POSTGRE_USER')
pgpwd = os.getenv('POSTGRE_PWD')
pgjar = os.getenv('POSTGRE_JAR')


In [4]:
#creating session

spark = (( SparkSession
            .builder
                .appName('customers')
                .config('spark.jar', f'{pgjar}')
                .config('spark.driver.extraClassPath', f'{pgjar}')
                .config('spark.executor.extraClassPath', f'{pgjar}')
            .getOrCreate()
        ))

your 131072x1 screen size is bogus. expect trouble
23/11/09 12:24:55 WARN Utils: Your hostname, DOUGPC resolves to a loopback address: 127.0.1.1; using 172.24.129.215 instead (on interface eth0)
23/11/09 12:24:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
23/11/09 12:24:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
#load customer data from source table

tmp_renamedrop_df = (( spark
                        .read
                        .format('jdbc')
                        .option('url', pgconn)
                        .option('driver', pgdriver)
                        .option('user', pguser)
                        .option('password', pgpwd)
                        .option('query', 
                                """ 
                                
                                    select

                                        customerid,
                                        customercategoryid,
                                        buyinggroupid,
                                        billtocustomerid,
                                        split_part(customername, '(', 1)as firstname,
		                                replace(split_part(customername, '(', 2), ')', '') as lastnameOrBranch,
                                        replace(split_part(customername, ',', 2),')', '') as place,
                                        accountopeneddate,
                                        creditlimit,
                                        standarddiscountpercentage,
                                        paymentdays,
                                        primarycontactpersonid,
                                        alternatecontactpersonid,
                                        deliverymethodid,
                                        deliverycityid,
                                        postalcityid,
                                        deliverypostalcode,
                                        deliveryaddressline1,
                                        deliveryaddressline2,
                                        postaladdressline1,
                                        postaladdressline2,
                                        isstatementsent,
                                        isoncredithold,
                                        _airbyte_ab_id,
                                        _airbyte_customers_hashid,
                                        _airbyte_emitted_at,
                                        _airbyte_normalized_at
                                        
                                    from sales.customers
                                
                                """)
                        
                    )).load().withColumnRenamed('_airbyte_ab_id', 'customer_elt_hashid')

In [6]:
#load customercategory data from source table
tmp_custmercategory = (( spark
                            .read
                            .format('jdbc')
                            .option('url', pgconn)
                            .option('driver', pgdriver)
                            .option('user', pguser)
                            .option('password', pgpwd)
                            .option('query', 
                                    
                                    """ 
                                        select

                                            customercategoryid,
                                            customercategoryname,
                                            _airbyte_ab_id,
                                            _airbyte_customercategories_hashid
                                        
                                        from sales.customercategories
                                        
                                    """)
                        )).load().withColumnRenamed('_airbyte_ab_id', 'custcategory_elt_hashid')

In [7]:
tmp_buyinggroups = (( spark
                        .read
                        .format('jdbc')
                        .option('url', pgconn)
                        .option('driver', pgdriver)
                        .option('user', pguser)
                        .option('password', pgpwd)
                        .option('query', 
                                
                                """
                                    select
                                    
                                        buyinggroupid,
                                        buyinggroupname,
                                        _airbyte_ab_id,
                                        _airbyte_buyinggroups_hashid
                                        
                                    from sales.buyinggroups
                                        
                                """)
                        )).load().withColumnRenamed('_airbyte_ab_id', 'custgroup_elt_hashid')

In [8]:
#joining dataframes

joined_frames = (( tmp_renamedrop_df
                    .join(tmp_custmercategory, 'customercategoryid', 'left')
                    .join(tmp_buyinggroups, 'buyinggroupid', 'left')
                ))

In [9]:
rename_df = ((
                joined_frames
                    .select('*', 
                            f.trim('firstname').alias('firstname_n'),
                            f.trim('lastnameOrbranch').alias('lastname'),
                            f.coalesce('creditlimit', f.lit(0.00)).alias('creditlimit_n'),
                            f.coalesce('buyinggroupid', f.lit(0)).alias('buyergroup_id'),
                            f.coalesce('alternatecontactpersonid', f.lit(0)).alias('alternatecontact_id')
                        )
))

In [10]:
renamedrop_df = ((rename_df
                     .drop('creditlimit')
                     .drop('buyinggroupid')
                     .drop('alternatecontactpersonid')
                     .drop('firstname')
                     .drop('lastnameOrBranch')
                     
                     #rename columns
                     .withColumnRenamed('creditlimit_n', 'creditlimit')
                     .withColumnRenamed('buyergroup_id', 'buyergroupid')
                     .withColumnRenamed('alternatecontact_id', 'alternatecontactid')
                     .withColumnRenamed('standarddiscountpercentage', 'discountpercentage')
                     .withColumnRenamed('primarycontactpersonid', 'primarycontactid')
                     .withColumnRenamed('deliverypostalcode', 'postalcode')
                     .withColumnRenamed('deliveryaddressline1', 'primaryaddress')
                     .withColumnRenamed('deliveryaddressline2', 'alternateaddress')
                     .withColumnRenamed('firstname_n', 'firstname')
                     .withColumnRenamed('postaladdressline1', 'postaladdress')
                     .withColumnRenamed('postaladdressline2', 'alternatepostaladdress')
                     .withColumnRenamed('_airbyte_customers_hashid', 'customers_hashid')
                     .withColumnRenamed('customer_elt_hashid', 'elt_customer_hashid')
                     .withColumnRenamed('_airbyte_customercategories_hashid', 'customercategory_hashid')
                     .withColumnRenamed('custcategory_elt_hashid', 'elt_custcategory_hashid')
                     .withColumnRenamed('_airbyte_buyinggroups_hashid', 'elt_buyergroup_hashid')
                     .withColumnRenamed('custgroup_elt_hashid', 'buyergroup_hashid')
                     .withColumnRenamed('_airbyte_emitted_at', 'elt_start')
                     .withColumnRenamed('_airbyte_normalized_at', 'elt_finalized')
                     
))

In [11]:
df = renamedrop_df.withColumn('lastname', f.split('lastname', '[^A-Za-z0-9 '']'))

renamedrop_df = df.withColumn('lastname', df['lastname'].getItem(0))


In [12]:
#organizing columns

customerframe= ((
                renamedrop_df
                    .select('customerid',
                            'customercategoryid',
                            'billtocustomerid',
                            'buyergroupid',
                            'firstname',
                            'lastname',
                            'customercategoryname',
                            'buyinggroupname',
                            'accountopeneddate',
                            'paymentdays',
                            'creditlimit',
                            'discountpercentage',
                            'primarycontactid',
                            'alternatecontactid',
                            'deliverymethodid',
                            'deliverycityid',
                            'postalcityid',
                            'postalcode',
                            'primaryaddress',
                            'alternateaddress',
                            'postaladdress',
                            'alternatepostaladdress',
                            'place',
                            'isoncredithold',
                            'isstatementsent',
                            'elt_customer_hashid',
                            'customers_hashid',
                            'elt_custcategory_hashid',
                            'customercategory_hashid',
                            'elt_buyergroup_hashid',
                            'buyergroup_hashid'
                            ).where('firstname not like "Tailspin%" and firstname not like "Wingtip%" ')
                    

))

In [18]:
#change boolean for 'binary' and empty for nulls
customer = ((
    customerframe
        .withColumn('isoncredithold', f.when(renamedrop_df['isoncredithold'] == 'true', 1).otherwise(0))
        .withColumn('isstatementsent', f.when(renamedrop_df['isstatementsent'] == 'true', 1).otherwise(0))
        .withColumn('buyinggroupname', f.expr('coalesce(buyinggroupname, "Personal Customer")'))
        .withColumn('place', f.when(renamedrop_df['place'] == "", None).otherwise(renamedrop_df['place']))
        
        #normalize null values without rename column
        .withColumn('place', f.expr('coalesce(place, "Not Available")'))
        .withColumn('elt_buyergroup_hashid', f.expr('coalesce(elt_buyergroup_hashid, "Not Available")'))
        .withColumn('buyergroup_hashid', f.expr('coalesce(buyergroup_hashid, "Not Available")'))
        .withColumnRenamed('firstname', 'customername')
        .drop('lastname')
)).show()

+----------+------------------+----------------+------------+--------------------+--------------------+-----------------+-----------------+-----------+-----------+------------------+----------------+------------------+----------------+--------------+------------+----------+--------------+--------------------+-------------+----------------------+-------------+--------------+---------------+--------------------+--------------------+-----------------------+-----------------------+---------------------+-----------------+
|customerid|customercategoryid|billtocustomerid|buyergroupid|        customername|customercategoryname|  buyinggroupname|accountopeneddate|paymentdays|creditlimit|discountpercentage|primarycontactid|alternatecontactid|deliverymethodid|deliverycityid|postalcityid|postalcode|primaryaddress|    alternateaddress|postaladdress|alternatepostaladdress|        place|isoncredithold|isstatementsent| elt_customer_hashid|    customers_hashid|elt_custcategory_hashid|customercategory_ha

In [ ]:
#create store table
storeframe = (( 
               renamedrop_df
                .select(    'customerid',
                            'customercategoryid',
                            'billtocustomerid',
                            'buyergroupid',
                            'firstname',
                            'lastname',
                            'customercategoryname',
                            'buyinggroupname',
                            'accountopeneddate',
                            'paymentdays',
                            'creditlimit',
                            'discountpercentage',
                            'primarycontactid',
                            'alternatecontactid',
                            'deliverymethodid',
                            'deliverycityid',
                            'postalcityid',
                            'postalcode',
                            'primaryaddress',
                            'alternateaddress',
                            'postaladdress',
                            'alternatepostaladdress',
                            'place',
                            'isoncredithold',
                            'isstatementsent',
                            'elt_customer_hashid',
                            'customers_hashid',
                            'elt_custcategory_hashid',
                            'customercategory_hashid',
                            'elt_buyergroup_hashid',
                            'buyergroup_hashid')
                .where('firstname like "Tailspin%" or firstname like "Wingtip%"')
                .withColumnRenamed('lastname', 'businessunity')
               ))

In [ ]:
#write to table

((
    customer
        .write
        .format('jdbc')
            .option('url', pgconn)
            .option('driver', pgdriver)
            .option('user', pguser)
            .option('password', pgpwd)
            .option('dbtable', 'sales.testeinsert')
        .save(mode= 'overwrite')
))

23/11/09 12:06:57 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [ ]:
#write to table

((
    tmp_custmercategory
        .write
        .format('jdbc')
            .option('url', pgconn)
            .option('driver', pgdriver)
            .option('user', pguser)
            .option('password', pgpwd)
            .option('dbtable', 'sales.testeinsert')
        .save(mode= 'overwrite')
))

In [ ]:
#write to table

((
    storeframe
        .write
        .format('jdbc')
            .option('url', pgconn)
            .option('driver', pgdriver)
            .option('user', pguser)
            .option('password', pgpwd)
            .option('dbtable', 'sales.testeinsert')
        .save(mode= 'overwrite')
))